**Import packages and libraries**

In [1]:
!pip uninstall -y numpy pandas

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2


In [2]:
!pip install "pandas==2.2.2"
!pip install "numpy==1.26.4"

  Using cached pandas-2.2.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached pandas-2.2.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.0 MB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_

In [3]:
!pip install gensim
!pip install scikit-learn matplotlib
!pip install transformers torch
!pip install datasets
!pip install -U sentence-transformers
!pip install -U accelerate
!pip install plotly
!pip install vaderSentiment accelerate
!pip install sentence-transformers
!pip install xgboost

In [4]:
# import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
import pandas as pd
import numpy as np
import json
import re

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

import networkx as nx
from networkx.algorithms import community
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, cohen_kappa_score, accuracy_score
from sklearn.utils import resample
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import xgboost as xgb

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x788e7702f9c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.11/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
     

**Import the git files**

In [6]:
!git clone https://github.com/coralie-sorbet/project_web_mining.git

Cloning into 'project_web_mining'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 376 (delta 6), reused 5 (delta 2), pack-reused 366 (from 1)
Receiving objects: 100% (376/376), 53.78 MiB | 13.69 MiB/s, done.
Resolving deltas: 100% (198/198), done.
Updating files: 100% (69/69), done.


In [7]:
path_data = '/content/project_web_mining/database/Everything/database_formated_for_NetworkX.graphml'

graph = nx.read_graphml(path_data)

**Extract types of nodes and priorities**

In [8]:
# Extract different kinds of nodes
unique_labels = set(data.get("labels") for _, data in graph.nodes(data=True))

print("Distincts types of nodes using 'labels' attribut:", unique_labels)

Distincts types of nodes using 'labels' attribut: {':Hashtag', ':User', ':Event', ':PostCategory', ':Tweet'}


**Priority Types**

In [9]:
# We want to see the different types of priority values of a tweet
priority_types = set(data.get("annotation_postPriority") for _, data in graph.nodes(data=True) if data.get("labels") == ":Tweet")

print(f"Number of unique values for 'annotation_postPriority': {len(priority_types)}")
print(f"Unique values of 'annotation_postPriority': {priority_types}")

Number of unique values for 'annotation_postPriority': 5
Unique values of 'annotation_postPriority': {'Critical', 'Medium', 'Low', 'Unknown', 'High'}


In [10]:
# We want to see the distribution of the priority values
tweet_priorities = [data['annotation_postPriority'] for _, data in graph.nodes(data=True) if data.get('labels') == ":Tweet"]

# Count occurrences of each annotation_postPriority
priority_counts = Counter(tweet_priorities)

# Display the counts
print(priority_counts)

Counter({'Low': 25407, 'Unknown': 19323, 'Medium': 6629, 'High': 4155, 'Critical': 472})


In [11]:
# We are only interested in the Low, Medium and High categories, so we can get rid of the Unknown and Critical ones
nodes_to_remove = [
    node for node, data in graph.nodes(data=True)
    if data.get('labels') == ":Tweet" and data.get('annotation_postPriority') in ["Unknown", "Critical"]
]
graph.remove_nodes_from(nodes_to_remove)

In [12]:
num_nodes = graph.number_of_nodes()
print(num_nodes)

89832


Get a random sample of 15,000 nodes to speed up analysis.

In [13]:
import random

random.seed(42)

# Get all nodes from the graph
all_nodes = list(graph.nodes())

# Select a random sample of 50,000 nodes (or all if fewer than 10,000 exist)
subset_size = min(15000, len(all_nodes))  # Avoid errors if graph has fewer nodes
random_nodes = random.sample(all_nodes, subset_size)

# Create a subgraph with only the selected nodes
graph = graph.subgraph(random_nodes).copy()


**Feature extraction and creation**

In [14]:
# Link tweets to user information
# Initialize lists to store data
tweet_ids = []
user_ids = []
user_names = []
followers_counts = []
listed = []
statuses = []
friends = []
verified = []
favourites = []
priorities = []
texts = []
favs = []
sens = []
annots = []
retweets = []
dates = []
screens = []

# Iterate over edges to extract information from "POSTED" relationships
for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "POSTED":
        tweet_id = graph.nodes[v].get("id_str")  # Tweet ID
        user_id = graph.nodes[u].get("id")  # User ID
        user_name = graph.nodes[u].get("name")  # User name
        user_favourites = graph.nodes[u].get("favourites_count", 0)
        user_listed = graph.nodes[u].get("listed_count", 0)
        user_friends = graph.nodes[u].get("friends_count", 0)
        user_statuses = graph.nodes[u].get("statuses_count", 0)
        user_verified = graph.nodes[u].get("isVerified")
        followers_count = graph.nodes[u].get("followers_count", 0)
        priority = graph.nodes[v].get("annotation_postPriority")
        screen_name = graph.nodes[u].get("screen_name")
        text = graph.nodes[v].get("text", "")
        sensitive = graph.nodes[v].get("possibly_sensitive")
        retweet = graph.nodes[v].get("retweet_count")
        fav = graph.nodes[v].get("favourite_count")
        annot = graph.nodes[v].get("annotation_num_judgements")
        date = graph.nodes[v].get("created_at")

        # Append data to lists
        tweet_ids.append(tweet_id)
        user_ids.append(user_id)
        user_names.append(user_name)
        followers_counts.append(followers_count)
        listed.append(user_listed)
        statuses.append(user_statuses)
        friends.append(user_friends)
        verified.append(user_verified)
        favourites.append(user_favourites)
        priorities.append(priority)
        texts.append(text)
        annots.append(annot)
        sens.append(sensitive)
        favs.append(fav)
        retweets.append(retweet)
        dates.append(date)
        screens.append(screen_name)

# Convert lists to DataFrame
df_tweets = pd.DataFrame({
    "tweet_id": tweet_ids,
    "user_id": user_ids,
    "user_name": user_names,
    "screen_name": screens,
    "listed_count": listed,
    "statuses_count": statuses,
    "friends_count": friends,
    "isVerified": verified,
    "favourites_count": favourites,
    "followers_count": followers_counts,
    "annotation_postPriority": priorities,
    "text": texts,
    "annotation_num_judgements": annot,
    "possibly_sensitive": sens,
    "retweet_count": retweet,
    "created_at": dates
})

# Link tweets to hashtags
tweet_ids_ht = []
hashtags_list = []
hashtags_count_list = []

for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "HAS_HASHTAG":
        tweet_id = graph.nodes[u].get("id_str")
        hashtag = graph.nodes[v].get("id")
        hashtag_count = graph.nodes[v].get("occurences", 0)

        tweet_ids_ht.append(tweet_id)
        hashtags_list.append(hashtag)
        hashtags_count_list.append(hashtag_count)

# Create DataFrame for hashtags
df_hashtags = pd.DataFrame({
    "tweet_id": tweet_ids_ht,
    "hashtag": hashtags_list,
    "hashtag_count": hashtags_count_list
})

# Link tweets to events
tweet_ids_event = []
event_type = []
event_ids = []

for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "IS_ABOUT":
        tweet_id_event = graph.nodes[u].get("id_str")
        event_id = graph.nodes[v].get("id")
        event = graph.nodes[v].get("eventType")

        tweet_ids_event.append(tweet_id_event)
        event_ids.append(event_id)
        event_type.append(event)

df_events = pd.DataFrame({
    "tweet_id": tweet_ids_event,
    "event_id": event_ids,
    "eventType": event_type
})

# Merge tweet, event, user and hashtag data using an outer join to preserve all tweets
df_final = pd.merge(df_tweets, df_hashtags, on="tweet_id", how="outer")
df_final = pd.merge(df_final, df_events, on="tweet_id", how="outer")
print(df_final.head())

              tweet_id             user_id                          user_name  \
0  1099981677659598848           472088946                         ABColombia   
1  1100063966955487232          3160419903               I am confusion Lizzy   
2  1100425366944935936                 NaN                                NaN   
3  1100622467733901312            39081817  Emergency Manager's Weekly Report   
4  1101201557465587715  943226635314135040                   Karakoles Travel   

       screen_name  listed_count  statuses_count  friends_count isVerified  \
0      ABColombia1          67.0          7562.0          796.0      False   
1  Echti_die_Echte           5.0         24183.0          218.0      False   
2              NaN           NaN             NaN            NaN        NaN   
3      emweeklyrpt         601.0         92438.0         1379.0      False   
4  karakolestravel           4.0           787.0         2242.0      False   

   favourites_count  followers_count annotat

In [15]:
# Mentions of users in a tweet
tweet_ids = []
user_mentioned_tweet = []
user_ids = []
mention_times = []
user_mentioned = []

for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "MENTIONS":
        if graph.nodes[u].get("labels") == ":Tweet":
            tweet_id = graph.nodes[u].get("id_str")
            tweet_ids.append(tweet_id)
            mentioned_user_id = graph.nodes[v].get("id")
            user_mentioned_tweet.append(mentioned_user_id)


for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "MENTIONS":
        if graph.nodes[u].get("labels") == ":User":
            mentioned_user_id = graph.nodes[v].get("id")
            user_mentioned.append(mentioned_user_id)
            user_id = graph.nodes[u].get("id")
            mention_time = edge_data.get("times")
            user_ids.append(user_id)
            mention_times.append(mention_time)

# Create DataFrame
df_mentions_tweet = pd.DataFrame({
    "tweet_id": tweet_ids,
    "mentioned_user": user_mentioned_tweet
})

df_mentions_user = pd.DataFrame({
    "user_id": user_ids,
    "mentioned_user": user_mentioned,
    "mention_times": mention_times
})

# Add the data on mentioned users in the tweet to our dataset
df_final = df_final.merge(df_mentions_tweet, on="tweet_id", how="outer")

In [16]:
df_final = df_final.drop_duplicates()

In [17]:
# Make sure the mention time is numeric
df_mentions_user.loc[:, "mention_times"] = pd.to_numeric(df_mentions_user["mention_times"], errors="coerce").astype("Int64")

# How many users someone mentions to identify users who interact with many different people
df_user_mentions_count = df_mentions_user.groupby("user_id")["mentioned_user"].nunique().reset_index()
df_user_mentions_count.rename(columns={"mentioned_user": "number_users_mentioned"}, inplace=True)

# Total mentions recieved: How often a user is mentioned
df_mentions_received = df_mentions_user.groupby("mentioned_user")["mention_times"].sum().reset_index()
df_mentions_received.rename(columns={"mention_times": "total_mentions_received"}, inplace=True)
df_mentions_received.rename(columns={"mentioned_user": "user_id"}, inplace=True)

# Mentions given vs recieved ratio
# > 1 : user mentions a lot of people but isn't mentioned much (marketer)
# < 1 : user is mentioned often but doesn't engage much (influencer)
df_mentions_given = df_mentions_user.groupby("user_id")["mention_times"].sum().reset_index()
df_mentions_given.rename(columns={"mention_times": "total_mentions_given"}, inplace=True)

df_mention_ratio = df_mentions_given.merge(df_mentions_received, on="user_id", how="outer").fillna(0)
df_mention_ratio["mention_ratio"] = df_mention_ratio["total_mentions_given"] / (df_mention_ratio["total_mentions_received"] + 1)
df_mention_ratio = df_mention_ratio[["user_id", "mention_ratio"]]

# Mutual mentions : high score = strong connections
df_reciprocal_mentions = df_mentions_user.merge(df_mentions_user, left_on=["user_id", "mentioned_user"], right_on=["mentioned_user", "user_id"])
df_reciprocal_mentions["reciprocity_score"] = df_reciprocal_mentions["mention_times_x"] + df_reciprocal_mentions["mention_times_y"]
df_reciprocal_mentions.rename(columns={"user_id_x": "user_id"}, inplace=True)
df_reciprocal_mentions = df_reciprocal_mentions[["user_id", "reciprocity_score"]]

# Merge all the data together
df_user_mentions_count = df_user_mentions_count.merge(df_mentions_received, on="user_id", how="outer").fillna(0)
df_user_mentions_count = df_user_mentions_count.merge(df_mentions_given, on="user_id", how="outer").fillna(0)
df_user_mentions_count = df_user_mentions_count.merge(df_mention_ratio, on="user_id", how="outer").fillna(0)
df_user_mentions_count = df_user_mentions_count.merge(df_reciprocal_mentions, on="user_id", how="outer").fillna(0)

df_final = pd.merge(df_final, df_user_mentions_count, on="user_id", how="outer")

In [18]:
# Create the column has_emoji

# Emoji pattern (covers most emojis)
emoji_pattern = re.compile("["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Symbols & pictographs
    "\U0001F680-\U0001F6FF"  # Transport & map symbols
    "\U0001F700-\U0001F77F"  # Alchemical symbols
    "\U0001F780-\U0001F7FF"  # Geometric shapes
    "\U0001F800-\U0001F8FF"  # Supplemental arrows
    "\U0001F900-\U0001F9FF"  # Supplemental symbols and pictographs
    "\U0001FA00-\U0001FA6F"  # Chess symbols, etc.
    "\U0001FA70-\U0001FAFF"  # More symbols
    "\U00002702-\U000027B0"  # Dingbats
    "\U000024C2-\U0001F251"
"]+", flags=re.UNICODE)

# Function to extract emojis from text
def extract_emojis(text):
    return emoji_pattern.findall(text) if isinstance(text, str) else []

# Apply functions to create new columns
df_final['has_emoji'] = df_final['text'].apply(lambda x: bool(emoji_pattern.search(str(x))))
df_final['emoji_count'] = df_final['text'].apply(lambda x: len(extract_emojis(str(x))))

In [ ]:
df_final.isin(["", None, np.nan]).sum()

,0
tweet_id,4391
user_id,1133
user_name,1528
screen_name,1528
listed_count,1528
statuses_count,1528
friends_count,1528
isVerified,1528
favourites_count,1528
followers_count,1528


In [19]:
df_final = df_final.dropna(subset = ['text', 'annotation_postPriority'])

In [ ]:
df_final.isin(["", None, np.nan]).sum()

,0
tweet_id,3996
user_id,0
user_name,0
screen_name,0
listed_count,0
statuses_count,0
friends_count,0
isVerified,0
favourites_count,0
followers_count,0


In [20]:
df_final['hashtag'] = df_final['hashtag'].fillna('None')
df_final['hashtag_count'] = df_final['hashtag_count'].fillna(0)

In [21]:
df_final[['number_users_mentioned', 'total_mentions_received',
          'total_mentions_given', 'mention_ratio', 'reciprocity_score']] = \
df_final[['number_users_mentioned', 'total_mentions_received',
          'total_mentions_given', 'mention_ratio', 'reciprocity_score']].fillna(0)


In [22]:
df_final['eventType'] = df_final['eventType'].fillna('Unrelated')
df_final['event_id'] = df_final['event_id'].fillna('Unrelated')

In [23]:
df_final['mentioned_user'] = df_final['mentioned_user'].fillna('No Mentions')

In [21]:
df_final.isin(["", None, np.nan]).sum()

,0
tweet_id,3996
user_id,0
user_name,0
screen_name,0
listed_count,0
statuses_count,0
friends_count,0
isVerified,0
favourites_count,0
followers_count,0


In [24]:
df_final = df_final.drop(columns= ['tweet_id', 'user_id'] )

In [25]:
# We want to detect opinianated words. We will use VADER Scores
# VADER gives four sentiment scores:
# pos (0 to 1), neg (0 to 1), neu (0 to 1), compound (-1 to 1)
# If compound > 0.05 :positive, if compound < -0.05 negative, if -0.05 < compound < 0.05 neutral

# Initialize the Sentiment Analyzer
sia = SentimentIntensityAnalyzer()

# Function to classify sentiment
def classify_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df_final['opinion'] = df_final['text'].apply(classify_sentiment)

In [5]:
import nltk
import gensim.downloader as api
from nltk.tokenize import word_tokenize

def get_word_embeddings():
  # Download and load a pre-trained Word2Vec model
  nltk.download('punkt_tab')
  model = api.load("word2vec-google-news-300")
  return model

def get_tweet_word_embedding(doc, model):
    def preprocess_text(text):
      return word_tokenize(text.lower())

    words = preprocess_text(doc)
    word_vectors = []
    for word in words:
        if word in model:
            word_vectors.append(model[word])

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    document_embedding = np.mean(word_vectors, axis=0)
    return document_embedding
w2v_model = get_word_embeddings()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [26]:
user_embeddings = []
usernames = df_final["user_name"]

for username in usernames:
    user_vector = get_tweet_word_embedding(username, w2v_model)
    user_embeddings.append(user_vector)

df_final["user_embeddings"] = user_embeddings

In [27]:
name_embeddings = []
screen_names = df_final["screen_name"]

for name in screen_names:
    user_vector = get_tweet_word_embedding(name, w2v_model)
    name_embeddings.append(user_vector)

df_final["screen_name_embeddings"] = name_embeddings

In [28]:
df_final = df_final.drop(columns= ['screen_name', 'user_name'] )

**Preparing the model**

In [29]:
X = df_final.drop('annotation_postPriority', axis=1)  # Features
y = df_final['annotation_postPriority']  # Target: the priority classes

# Initialize Stratified Shuffle Split (to maintain the class distribution in both train and test)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

# Split data
for train_index, test_index in splitter.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Combine features and target into a train/test dataset
train_data = X_train.copy()
train_data['annotation_postPriority'] = y_train

test_data = X_test.copy()
test_data['annotation_postPriority'] = y_test


In [30]:
# Check class distribution in the training and test sets
print("Training Set Class Distribution:")
print(train_data['annotation_postPriority'].value_counts())

print("\nTest Set Class Distribution:")
print(test_data['annotation_postPriority'].value_counts())


Training Set Class Distribution:
annotation_postPriority
Low       3085
Medium     338
High        74
Name: count, dtype: int64

Test Set Class Distribution:
annotation_postPriority
Low       1323
Medium     145
High        31
Name: count, dtype: int64


In [31]:
# Separate the majority and minority classes in the training data
majority_class = train_data[train_data['annotation_postPriority'] == 'Low']
minority_class = train_data[train_data['annotation_postPriority'] != 'Low']

# Oversample the minority class
minority_upsampled = resample(minority_class,
                               replace=True,
                               n_samples=len(majority_class),  # Match the size of the majority class
                               random_state=42)

# Combine majority class with oversampled minority class
train_balanced = pd.concat([majority_class, minority_upsampled])

# Shuffle the training data to ensure randomness
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


In [32]:
# Check class distribution in the training and test sets
print("Training Set Class Distribution:")
print(train_balanced['annotation_postPriority'].value_counts())

print("\nTest Set Class Distribution:")
print(test_data['annotation_postPriority'].value_counts())


Training Set Class Distribution:
annotation_postPriority
Low       3085
Medium    2513
High       572
Name: count, dtype: int64

Test Set Class Distribution:
annotation_postPriority
Low       1323
Medium     145
High        31
Name: count, dtype: int64


In [33]:
X_train_resampled = train_balanced.drop('annotation_postPriority', axis=1)
y_train_resampled = train_balanced['annotation_postPriority']

# Drop non-numeric columns
X_train_numeric = X_train_resampled.select_dtypes(exclude=['object'])
X_test_numeric = X_test.select_dtypes(exclude=['object'])  # Assuming X_test comes from the original test set

# Scale the numeric features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)  # Fit and transform on train data
X_test_scaled = scaler.transform(X_test_numeric)  # Transform on test data


# Apply TF-IDF on the 'text' column
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_text = tfidf_vectorizer.fit_transform(X_train_resampled['text']).toarray()
X_test_text = tfidf_vectorizer.transform(X_test['text']).toarray()

# Now 'text' is numerically represented, we can concatenate it with the rest
X_train_final = np.concatenate([X_train_scaled, X_train_text], axis=1)
X_test_final = np.concatenate([X_test_scaled, X_test_text], axis=1)


**Prediction Models**

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score
from tqdm import tqdm

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

# Convert param_grid into a list of all possible parameter combinations
param_combinations = list(ParameterGrid(param_grid))

# Initialize variables to track the best model
best_f1 = 0
best_params = None

# Loop over all parameter combinations with a progress bar
for params in tqdm(param_combinations, desc="Grid Search Progress"):
    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train_final, y_train_resampled)  # Train the model

    y_pred = model.predict(X_test_final)  # Make predictions
    f1 = f1_score(y_test, y_pred, average='weighted')  # Compute weighted F1-score

    if f1 > best_f1:  # Keep track of the best F1-score
        best_f1 = f1
        best_params = params

# Print best parameters
print("Best Parameters:", best_params)
print("Best F1 Score:", best_f1)


Grid Search Progress:   1%|          | 2/162 [00:13<17:35,  6.60s/it]


KeyboardInterrupt: 

In [34]:
# RandomForestClassifier
model = RandomForestClassifier(
    criterion='entropy',
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=10,
    n_estimators=200,
    random_state=42
)

model.fit(X_train_final, y_train_resampled)

y_pred = model.predict(X_test_final)

print("\nClassification Report:\n", classification_report(y_test, y_pred))
f1_random_class = f1_score(y_test, y_pred, average='weighted')


Classification Report:
               precision    recall  f1-score   support

        High       0.33      0.06      0.11        31
         Low       0.95      0.99      0.97      1323
      Medium       0.89      0.70      0.79       145

    accuracy                           0.95      1499
   macro avg       0.73      0.59      0.62      1499
weighted avg       0.94      0.95      0.94      1499



In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm

# Define the parameter grid
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', None],  # Regularization type
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Regularization strength
    'solver': ['liblinear', 'lbfgs', 'saga'],  # Optimization solvers
}

# Initialize Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Convert param_grid into a list of all possible parameter combinations
param_combinations = list(ParameterGrid(param_grid))

# Initialize variables to track the best F1 score and parameters
best_f1 = 0
best_params = None

# Loop over all parameter combinations with a progress bar
for params in tqdm(param_combinations, desc="Grid Search Progress"):
    try:
        # Initialize and train the Logistic Regression model
        model = LogisticRegression(**params, max_iter=1000, random_state=42)
        model.fit(X_train_final, y_train_resampled)

        # Make predictions on the test set
        y_pred = model.predict(X_test_final)

        # Calculate the F1 score
        f1 = f1_score(y_test, y_pred, average='weighted')  # Weighted average for multi-class problems

        # If the current F1 score is better, update best score and parameters
        if f1 > best_f1:
            best_f1 = f1
            best_params = params

    except Exception as e:
        print(f"Skipping parameters {params} due to error: {e}")

# Print the best F1 score and corresponding parameters
print("Best Parameters:", best_params)
print("Best F1 Score:", best_f1)


Grid Search Progress:   1%|▏         | 1/72 [00:00<00:29,  2.43it/s]

Skipping parameters {'C': 0.001, 'penalty': 'l1', 'solver': 'lbfgs'} due to error: Solver lbfgs supports only 'l2' or None penalties, got l1 penalty.


Grid Search Progress:   8%|▊         | 6/72 [00:17<05:02,  4.58s/it]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Skipping parameters {'C': 0.001, 'penalty': 'elasticnet', 'solver': 'liblinear'} due to error: Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Skipping parameters {'C': 0.001, 'penalty': 'elasticnet', 'solver': 'lbfgs'} due to error: Solver lbfgs supports only 'l2' or None penalties, got elasticnet penalty.
Skipping parameters {'C': 0.001, 'penalty': 'elasticnet', 'solver': 'saga'} due to error: l1_ratio must be specified when penalty is elasticnet.
Skipping parameters {'C': 0.001, 'penalty': None, 'solver': 'liblinear'} due to error: penalty=None is not supported for the liblinear solver


Grid Search Progress:  15%|█▌        | 11/72 [00:34<03:47,  3.74s/it]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Grid Search Progress:  18%|█▊        | 13/72 [10:08<1:18:12, 79.53s/it]

...

Skipping parameters {'C': 100, 'penalty': 'l1', 'solver': 'lbfgs'} due to error: Solver lbfgs supports only 'l2' or None penalties, got l1 penalty.


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Grid Search Progress:  90%|█████████ | 65/72 [2:30:58<21:39, 185.64s/it]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Grid Search Progress:  92%|█████████▏| 66/72 [2:40:34<27:49, 278.29s/it]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Skipping parameters {'C': 100, 'penalty': 'elasticnet', 'solver': 'liblinear'} due to error: Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Skipping parameters {'C': 100, 'penalty': 'elasticnet', 'solver': 'lbfgs'} due to error: Solver lbfgs supports only 'l2' or None penalties, got elasticnet penalty.
Skipping parameters {'C': 100, 'penalty': 'elasticnet', 'solver': 'saga'} due to error: l1_ratio must be specified when penalty is elasticnet.
Skipping parameters {'C': 100, 'penalty': None, 'solver': 'liblinear'} due to error: penalty=None is not supported for the liblinear solver


Grid Search Progress:  99%|█████████▊| 71/72 [2:40:50<01:45, 105.30s/it]/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1207: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Grid Search Progress: 100%|██████████| 72/72 [2:50:26<00:00, 142.03s/it]

Best Parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best F1 Score: 0.9298924468616412


In [35]:
# Logistic Regression
log_reg = LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=1000, random_state=42)

log_reg.fit(X_train_final, y_train_resampled)

y_pred_log = log_reg.predict(X_test_final)

print("\nClassification Report:\n", classification_report(y_test, y_pred_log))
f1_log_reg = f1_score(y_test, y_pred_log, average='weighted')


Classification Report:
               precision    recall  f1-score   support

        High       0.28      0.23      0.25        31
         Low       0.97      0.97      0.97      1323
      Medium       0.72      0.76      0.74       145

    accuracy                           0.93      1499
   macro avg       0.66      0.65      0.65      1499
weighted avg       0.93      0.93      0.93      1499



In [36]:
from sklearn.svm import SVC
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm import tqdm

# Define the parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],  # Regularization parameter
    'gamma': [0.001, 0.01, 0.1, 1],  # Kernel coefficient
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid']  # Kernel type (already 'rbf' in the model)
}

# Initialize variables to track the best F1 score and parameters
best_f1 = 0
best_params = None

# Convert param_grid into a list of all possible parameter combinations
param_combinations = list(ParameterGrid(param_grid))

# Loop over all parameter combinations with a progress bar
for params in tqdm(param_combinations, desc="Grid Search Progress"):
    try:
        # Initialize and train the SVM model
        svm_model = SVC(**params, random_state=42)
        svm_model.fit(X_train_final, y_train_resampled)

        # Make predictions on the test set
        y_pred_svm = svm_model.predict(X_test_final)

        # Calculate the F1 score
        f1 = f1_score(y_test, y_pred_svm, average='weighted')  # Weighted average for multi-class problems

        # If the current F1 score is better, update best score and parameters
        if f1 > best_f1:
            best_f1 = f1
            best_params = params

    except Exception as e:
        print(f"Skipping parameters {params} due to error: {e}")

# Print the best F1 score and corresponding parameters
print("Best Parameters:", best_params)
print("Best F1 Score:", best_f1)

best_svm_model = SVC(**best_params, random_state=42)
best_svm_model.fit(X_train_final, y_train_resampled)
y_pred_best_svm = best_svm_model.predict(X_test_final)

# Print the classification report for the best model
print("\nClassification Report for Best SVM Model:\n", classification_report(y_test, y_pred_best_svm))

f1_svm = f1_score(y_test, y_pred_best_svm, average='weighted')  # Weighted average for multi-class problems


Grid Search Progress: 100%|██████████| 64/64 [1:11:03<00:00, 66.61s/it]

Best Parameters: {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
Best F1 Score: 0.936875806868144


ValueError: Found input variables with inconsistent numbers of samples: [6170, 3497]

In [38]:

best_svm_model = SVC(C = 10, gamma = 0.1, kernel = 'rbf', random_state=42)
best_svm_model.fit(X_train_final, y_train_resampled)
y_pred_best_svm = best_svm_model.predict(X_test_final)

# Print the classification report for the best model
print("\nClassification Report for Best SVM Model:\n", classification_report(y_test, y_pred_best_svm))

f1_svm = f1_score(y_test, y_pred_best_svm, average='weighted')  # Weighted average for multi-class problems



Classification Report for Best SVM Model:
               precision    recall  f1-score   support

        High       0.50      0.16      0.24        31
         Low       0.95      0.99      0.97      1323
      Medium       0.86      0.69      0.77       145

    accuracy                           0.94      1499
   macro avg       0.77      0.61      0.66      1499
weighted avg       0.94      0.94      0.94      1499



In [37]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm  # Import tqdm for progress bar

# Initialize label encoder
label_encoder = LabelEncoder()
# Encode the target labels
y_train_encoded = label_encoder.fit_transform(y_train_resampled)
y_test_encoded = label_encoder.transform(y_test)

# XGBoost Classifier
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma': [0, 0.1]
}
# Convert param_grid into a list of all possible parameter combinations
param_combinations = list(ParameterGrid(param_grid))

# Initialize variables to track the best F1 score and parameters
best_f1 = 0
best_params = None

# Manually loop over all parameter combinations with a progress bar
for params in tqdm(param_combinations, desc="Grid Search Progress"):
    try:
        # Initialize and train the XGBoost model
        xgboost_model = xgb.XGBClassifier(**params, use_label_encoder=False, eval_metric="logloss", random_state=42)
        xgboost_model.fit(X_train_final, y_train_encoded)

        # Make predictions on the test set
        y_pred_xgboost = xgboost_model.predict(X_test_final)

        # Calculate the F1 score
        f1 = f1_score(y_test_encoded, y_pred_xgboost, average='weighted')  # Weighted average for multi-class problems

        # If the current F1 score is better, update best score and parameters
        if f1 > best_f1:
            best_f1 = f1
            best_params = params

    except Exception as e:
        print(f"Skipping parameters {params} due to error: {e}")

# Print the best F1 score and corresponding parameters
print("Best Parameters:", best_params)
print("Best F1 Score:", best_f1)

# Use the best model to predict on the test set
best_xgb_model = xgb.XGBClassifier(**best_params, use_label_encoder=False, eval_metric="logloss", random_state=42)
best_xgb_model.fit(X_train_final, y_train_encoded)
y_pred_best_xgboost = best_xgb_model.predict(X_test_final)

# Print the classification report for the best model
print("\nClassification Report for Best XGBoost Model:\n", classification_report(y_test_encoded, y_pred_best_xgboost))
f1_xgb = f1_score(y_test_encoded, y_pred_best_xgboost, average='weighted')  # Weighted average for multi-class problems

Grid Search Progress:   0%|          | 0/128 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:36:15] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Grid Search Progress:   1%|          | 1/128 [00:23<48:56, 23.12s/it]/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:36:36] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Grid Search Progress:   2%|▏         | 2/128 [00:40<41:05, 19.57s/it]/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:36:52] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
Grid Search Progress:   2%|▏         | 3/128 [01:10<51:21, 24.65s/it]/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:37:23] WARNING: /w

Best Parameters: {'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 200, 'subsample': 0.8}
Best F1 Score: 0.9167248394920059


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [13:26:32] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



Classification Report for Best XGBoost Model:
               precision    recall  f1-score   support

           0       0.13      0.13      0.13        31
           1       0.97      0.95      0.96      1323
           2       0.64      0.76      0.69       145

    accuracy                           0.91      1499
   macro avg       0.58      0.61      0.59      1499
weighted avg       0.92      0.91      0.92      1499

